# HLLSet as the K-Space of Attention — Interactive Demo

This notebook runs in the **ewm-cortex** workspace and demonstrates the design from `_DOCS/arch/HLLSET_LUT_TRANSFORMER_ARCHITECTURE.md`
and `_DOCS/dev/HLLSET_K_SPACE_MATH.md`:

1. **K-storage** — `TokenLutStorage` (tier 1) and `CatalogLutStorage` (tier 2) as the K side of attention.
2. **Collision statistics** — the HLLSet partition's pairwise collision rate `p = 1/3072`.
3. **Mode A address-key** — `KBridge` + `BitKeyTable`: one key vector per bit cell.
4. **Bounded vocabulary** — `ContextVocabulary`: `V(t) = (V \ M(D)) ∪ M(N)` with a hash gate.
5. **Phase 0/1 transformer** — the hand-rolled reproduction transformer in the token realm.


In [2]:
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex/crates/hllset-core" }
:dep hllset-attn = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex/crates/hllset-attn" }
:dep hllset-repro = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex/crates/hllset-repro" }

use hllset_attn::*;
use hllset_core::HLLSet;

println!("crates loaded: hllset-core, hllset-attn, hllset-repro");


crates loaded: hllset-core, hllset-attn, hllset-repro


---
## 1. K-storage — the LUT as K

Every token hashes to exactly one `<reg, tz>` cell: `bit = reg * 32 + tz`.
`TokenLutStorage` resolves tokens to keys (`key_of`), keys to candidate
tokens (`candidates`, the V side), and reports LUT coverage (`confidence`).


In [3]:
let storage = TokenLutStorage::from_tokens(&["hello", "world", "lattice"]);

for token in ["hello", "world", "lattice"] {
    match storage.key_of(token.as_bytes()) {
        KeyRef::Cell(bit) => println!("{token:>8} -> bit {bit:>5} (reg {}, tz {})", bit / 32, bit % 32),
        _ => unreachable!("tier 1 keys are single cells"),
    }
}

let hllset = HLLSet::from_tokens(&["hello", "world", "lattice"]);
println!("candidates : {:?}", storage.candidates(&hllset));
println!("confidence : {:.3} (1.0 = coverage invariant holds)", storage.confidence(&hllset));


   hello -> bit 24641 (reg 770, tz 1)
   world -> bit  7488 (reg 234, tz 0)
 lattice -> bit 16544 (reg 517, tz 0)
candidates : [[104, 101, 108, 108, 111], [119, 111, 114, 108, 100], [108, 97, 116, 116, 105, 99, 101]]
confidence : 1.000 (1.0 = coverage invariant holds)


In [4]:
// Tier 2: multi-seed quorum (3 seeds, 2-of-3 consensus).
let catalog = CatalogLutStorage::from_values(&["alice@example.com", "bob@example.com"]);

match catalog.key_of(b"alice@example.com") {
    KeyRef::Cells(cells, quorum) => println!("alice -> {} cells, quorum {}: {:?}", cells.len(), quorum, cells),
    _ => unreachable!("tier 2 keys are multi-seed"),
}


alice -> 3 cells, quorum 2: [25760, 25953, 9857]


()

---
## 2. Collision statistics — theory vs observation

The trailing-zero distribution is geometric, so two random tokens collide
with probability `p = 1/3072`. For a vocabulary of `n` tokens, the expected
number of colliding pairs is `C(n,2) / 3072`.


In [5]:
use std::collections::HashMap;

let tokens: Vec<Vec<u8>> = (0..5000).map(|i| format!("w{i}").into_bytes()).collect();
let storage = TokenLutStorage::from_tokens(tokens.iter());

let mut groups: HashMap<u32, usize> = HashMap::new();
for t in &tokens {
    let bit = storage.key_of(t).cells()[0];
    *groups.entry(bit).or_insert(0) += 1;
}
let observed_pairs: u64 = groups.values().map(|&g| (g * (g - 1) / 2) as u64).sum();
let expected_pairs = (tokens.len() * (tokens.len() - 1)) as f64 / 2.0 / 3072.0;

println!("vocab         : {}", tokens.len());
println!("occupied cells: {}", groups.len());
println!("observed pairs: {observed_pairs}");
println!("expected pairs: {expected_pairs:.2}");
println!("observed prob : {:.6}", 2.0 * observed_pairs as f64 / (tokens.len() * (tokens.len() - 1)) as f64);
println!("theory prob   : {:.6}", 1.0 / 3072.0);


vocab         : 5000
occupied cells: 2754
observed pairs: 3942
expected pairs: 4068.20
observed prob : 0.000315
theory prob   : 0.000326


---
## 3. Mode A — address-key attention

`KBridge` routes a token through K-storage into a key vector. Tier 1 reads
the token's single cell from the learned `BitKeyTable` (32,768 × d); tier 2
returns the centroid of its multi-seed cells.


In [6]:
let storage = TokenLutStorage::from_tokens(&["hello"]);
let bridge = KBridge::new(storage);

let bit = match bridge.key_ref(b"hello") {
    KeyRef::Cell(b) => b,
    other => panic!("expected Cell, got {other:?}"),
};

let mut table = BitKeyTable::new(4);
table.set(bit, &[0.5, -0.25, 1.0, 0.75]);
println!("E_bit[{bit}] = {:?}", bridge.key_vector(b"hello", &table));

// residual add (positional/contextual term)
let key = bridge.key_vector_with(b"hello", &table, &[0.1, 0.2, -0.3, 0.4]);
println!("key + residual = {key:?}");


E_bit[24641] = [0.5, -0.25, 1.0, 0.75]
key + residual = [0.6, -0.049999997, 0.7, 1.15]


---
## 4. Bounded vocabulary — the hash gate over the LUT

`ContextVocabulary` maintains `V(t)` incrementally:
`V(t) = (V(t-1) \ M(D)) ∪ M(N)`. The mask stores content addresses
(murmur3 hashes), never token bytes — bytes live only in the LUT, so the
vocabulary view cannot drift out of sync.


In [7]:
let sentences = [
    "the cat sat",
    "the dog ran",
    "the cat purred",
];

let mut vocab = ContextVocabulary::new(TokenLutStorage::new());
for (i, s) in sentences.iter().enumerate() {
    let tokens: Vec<Vec<u8>> = s.split_whitespace().map(|w| w.as_bytes().to_vec()).collect();
    for t in &tokens { vocab.register(t.clone()); }
    let delta = vocab.accumulate(&HLLSet::from_tokens(tokens.iter()));
    println!("step {i}: |V(t)| = {:>2}  added = {:?}  coverage = {:.2}",
        vocab.len(), delta.added, vocab.coverage());
}
println!("final vocabulary: {:?}", vocab.tokens());


step 0: |V(t)| =  3  added = [[99, 97, 116], [115, 97, 116], [116, 104, 101]]  coverage = 1.00
step 1: |V(t)| =  5  added = [[100, 111, 103], [114, 97, 110]]  coverage = 1.00
step 2: |V(t)| =  6  added = [[112, 117, 114, 114, 101, 100]]  coverage = 1.00
final vocabulary: [[112, 117, 114, 114, 101, 100], [100, 111, 103], [115, 97, 116], [116, 104, 101], [114, 97, 110], [99, 97, 116]]


---
## 5. Phase 0/1 — reproduction transformer (token realm)

A hand-rolled character-level transformer (`hllset-repro`) trains in the
token realm; Phase 1 attaches the K-storage and verifies the invariants.


In [8]:
use hllset_repro::{attach, CharDataset};

let dataset = CharDataset::from_text(hllset_repro::CORPUS);
let report = attach(&dataset);
println!("corpus vocab   : {} characters", report.vocab_size);
println!("coverage       : {:.3} {}", report.coverage, if report.coverage == 1.0 { "✓ invariant" } else { "✗" });
println!("synthetic 5000 : observed {} pairs vs expected {:.2} — {}",
    report.synthetic.observed_pairs, report.synthetic.expected_pairs,
    if report.synthetic_matches_theory { "✓ matches theory" } else { "✗" });


corpus vocab   : 23 characters
coverage       : 1.000 ✓ invariant
synthetic 5000 : observed 3942 pairs vs expected 4068.20 — ✓ matches theory


In [9]:
{
use hllset_repro::{Adam, Config, Transformer, XorShift};

let dataset = CharDataset::from_text(hllset_repro::CORPUS);
let cfg = Config::small(dataset.vocab_size(), 32);
let model = Transformer::new(cfg, 42);
let mut opt = Adam::new(1e-3);
let data = dataset.data().to_vec(); // owned Vec
let mut rng = XorShift::new(7);

for step in 0..600 {
    let start = rng.index(data.len() - 32 - 1);
    let inputs = data[start..start + 32].to_vec();
    let targets = data[start + 1..start + 33].to_vec();
    let loss = model.train_step(&mut opt, &inputs, &targets);
    if (step + 1) % 150 == 0 {
        println!("step {:>4}: loss {:.4}", step + 1, loss);
    }
}

let prompt = dataset.encode("the cat");
let sampled = model.sample(&prompt, 60);
println!("sample: {}", dataset.decode(&sampled));
}


step  150: loss 0.9232
step  300: loss 0.9071
step  450: loss 0.7448
step  600: loss 0.9680
sample: the cat and the ran rat rat ran from the from t from the ran fox an


()

---
## Summary

- **K is a routing address.** The only invariant is the similarity geometry; any mutually
  exclusive vocabulary partition is a valid K-representation.
- **The HLLSet partition** (`reg`, geometric `tz`) gives a multiresolution K-space with
  pairwise collision `p = 1/3072`; multi-seed quorum makes it exact.
- **Two gates over one LUT:** the HLLSet context (bit gate, routing) and the `TokenMask`
  (hash gate, exact vocabulary). Bytes exist once, in the LUT.
- **The transformer stays in the token realm** (Q, V, FFN, output); K is served by the
  HLLSet realm through `KBridge`.
